# Data Cleansing Pipeline for Large Dataset

This notebook implements a data processing pipeline to cleanse 5,000,000 rows of raw data from the USA spending prime awards table while preserving the original data. The notebook covers:

1. Connecting to the PostgreSQL database
2. Loading a subset of raw data
3. Performing data cleansing operations
4. Saving the cleansed data for further processing

## 1. Connect to PostgreSQL Database

First, we'll import the necessary libraries and establish a connection to our PostgreSQL database.

In [ ]:
# Import required libraries for database connection and data manipulation
import pandas as pd
import numpy as np
import psycopg2
from sqlalchemy import create_engine
import time
import os
from datetime import datetime

# For visualizing data sample and statistics
import matplotlib.pyplot as plt
import seaborn as sns

# Configure pandas to display more columns
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

In [ ]:
# Database connection parameters
# You should replace these with your actual database credentials
db_params = {
    'dbname': 'usa_spending_db',
    'user': 'db_user',
    'password': 'your_password',
    'host': 'localhost',
    'port': '5432'
}

# Create SQLAlchemy engine for pandas operations
engine_string = f"postgresql://{db_params['user']}:{db_params['password']}@{db_params['host']}:{db_params['port']}/{db_params['dbname']}"
engine = create_engine(engine_string)

# Test the connection
try:
    # Create a direct connection with psycopg2 for raw SQL operations if needed
    conn = psycopg2.connect(**db_params)
    cursor = conn.cursor()
    print("Database connection established successfully!")
    
    # Get list of tables in the database
    cursor.execute("SELECT table_name FROM information_schema.tables WHERE table_schema = 'public'")
    tables = cursor.fetchall()
    print("\nAvailable tables:")
    for table in tables:
        print(f"- {table[0]}")
    
except Exception as e:
    print(f"Error connecting to the database: {e}")
    raise

## 2. Load a Subset of Raw Data

Now we'll query the database to load 5,000,000 rows from the raw data table into a Pandas DataFrame.
Since this is a large amount of data, we'll use efficient loading techniques:

1. Use `LIMIT` in the SQL query to restrict the number of rows
2. Select only necessary columns to reduce memory usage
3. Specify data types where possible to optimize memory

In [ ]:
# Function to calculate memory usage of a DataFrame
def memory_usage(df):
    memory_usage_bytes = df.memory_usage(deep=True).sum()
    memory_usage_mb = memory_usage_bytes / (1024 * 1024)
    return f"Memory usage: {memory_usage_mb:.2f} MB"

# Start timing the data loading
start_time = time.time()

# Query to load 5 million rows from the raw data table
# Adjust the column list based on your actual table structure
query = """
SELECT 
    award_id, 
    generated_pragmatic_obligation, 
    award_type, 
    award_type_description,
    action_date,
    action_type,
    action_type_description,
    modification_number,
    contract_award_unique_key,
    total_obligation,
    total_subsidy_cost,
    awarding_agency_id,
    funding_agency_id,
    recipient_uei,
    recipient_name,
    recipient_city_name,
    recipient_county_name,
    recipient_state_code,
    recipient_zip_code,
    place_of_performance_city,
    place_of_performance_state,
    place_of_performance_zip,
    award_description
FROM usaspening_prime_awards
LIMIT 5000000
"""

# Load data into a DataFrame
try:
    print("Loading data from database... This may take several minutes.")
    df_raw = pd.read_sql(query, engine)
    
    # Calculate and display loading time
    load_time = time.time() - start_time
    print(f"Data loaded successfully in {load_time:.2f} seconds")
    print(f"Loaded {df_raw.shape[0]} rows and {df_raw.shape[1]} columns")
    print(memory_usage(df_raw))
    
    # Display sample data and summary statistics
    print("\nData Sample:")
    display(df_raw.head())
    
    # Get data types and null counts
    print("\nData Types and Null Counts:")
    display(pd.DataFrame({
        'Data Type': df_raw.dtypes,
        'Null Count': df_raw.isnull().sum(),
        'Null Percentage': (df_raw.isnull().sum() / len(df_raw) * 100).round(2)
    }))

except Exception as e:
    print(f"Error loading data: {e}")
    raise

## 3. Data Cleansing

Now we'll perform various data cleansing operations on our dataset. This includes:

1. Handling missing values
2. Correcting data types
3. Removing invalid entries
4. Standardizing text fields
5. Validating numeric values
6. Creating a data quality summary

We'll implement these steps systematically, keeping track of changes made to ensure transparency.

In [ ]:
# Create a copy of the raw data to preserve the original while we clean
df_clean = df_raw.copy()

# Create a data quality log to track changes
quality_log = {
    'operation': [],
    'columns_affected': [],
    'rows_affected': [],
    'description': []
}

# Function to add an entry to our data quality log
def log_operation(operation, columns, rows_affected, description):
    quality_log['operation'].append(operation)
    quality_log['columns_affected'].append(columns)
    quality_log['rows_affected'].append(rows_affected)
    quality_log['description'].append(description)
    
    # Print summary of the operation
    print(f"{operation}: {description} ({rows_affected} rows affected)")

In [ ]:
# Step 1: Convert date fields to datetime type
date_columns = ['action_date']

for col in date_columns:
    if col in df_clean.columns:
        # Count invalid dates before conversion
        invalid_before = df_clean[~df_clean[col].astype(str).str.match(r'^\d{4}-\d{2}-\d{2}$')].shape[0]
        
        # Convert to datetime
        df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
        
        # Count NaT values after conversion
        invalid_after = df_clean[col].isna().sum()
        
        log_operation(
            'Date Conversion', 
            col, 
            invalid_after, 
            f"Converted {col} to datetime. {invalid_after - invalid_before} additional invalid values identified."
        )

In [ ]:
# Step 2: Convert numeric fields to appropriate types
# Identify numeric columns
numeric_columns = [
    'generated_pragmatic_obligation', 
    'total_obligation', 
    'total_subsidy_cost'
]

for col in numeric_columns:
    if col in df_clean.columns:
        # Store original null count
        original_nulls = df_clean[col].isna().sum()
        
        # Convert to numeric
        df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce')
        
        # Count new nulls
        new_nulls = df_clean[col].isna().sum()
        invalid_values = new_nulls - original_nulls
        
        log_operation(
            'Numeric Conversion', 
            col, 
            invalid_values, 
            f"Converted {col} to numeric. {invalid_values} invalid values identified and converted to NaN."
        )

In [ ]:
# Step 3: Handle missing values
# For each column, decide how to handle missing values based on the column's purpose

# For ID columns - can't impute, but can flag records with missing IDs
id_columns = ['award_id', 'contract_award_unique_key', 'recipient_uei']
for col in id_columns:
    if col in df_clean.columns:
        missing_ids = df_clean[col].isna().sum()
        if missing_ids > 0:
            # Create a flag column for missing IDs
            df_clean[f'{col}_missing'] = df_clean[col].isna()
            
            log_operation(
                'Missing ID Flagging', 
                col, 
                missing_ids,
                f"Created flag column for missing {col} values."
            )

# For categorical columns - replace with "Unknown" or similar value
categorical_columns = [
    'award_type', 'award_type_description', 'action_type', 
    'action_type_description', 'recipient_name', 'recipient_city_name',
    'recipient_county_name', 'recipient_state_code', 'place_of_performance_city',
    'place_of_performance_state'
]

for col in categorical_columns:
    if col in df_clean.columns:
        missing_values = df_clean[col].isna().sum()
        if missing_values > 0:
            # Replace NaNs with "Unknown"
            df_clean[col] = df_clean[col].fillna('Unknown')
            
            log_operation(
                'Missing Value Imputation', 
                col, 
                missing_values,
                f"Replaced missing values with 'Unknown' in categorical column {col}."
            )

# For numeric columns - consider the context for imputation strategy
# Here we'll replace with 0 for financial values where missing likely means no value
financial_columns = ['total_obligation', 'total_subsidy_cost']
for col in financial_columns:
    if col in df_clean.columns:
        missing_values = df_clean[col].isna().sum()
        if missing_values > 0:
            # Replace NaNs with 0
            df_clean[col] = df_clean[col].fillna(0)
            
            log_operation(
                'Missing Value Imputation', 
                col, 
                missing_values, 
                f"Replaced missing values with 0 in financial column {col}."
            )

In [ ]:
# Step 4: Standardize text fields
# For text fields, we'll standardize by:
# 1. Converting to lowercase
# 2. Trimming whitespace
# 3. Standardizing certain patterns

text_columns = ['recipient_name', 'award_description']

for col in text_columns:
    if col in df_clean.columns:
        # Count records before standardization
        before_count = df_clean[col].nunique()
        
        # Apply standardization
        df_clean[col] = df_clean[col].astype(str).str.strip().str.lower()
        
        # Count records after standardization
        after_count = df_clean[col].nunique()
        
        log_operation(
            'Text Standardization', 
            col, 
            before_count - after_count, 
            f"Standardized text in {col}. Reduced unique values from {before_count} to {after_count}."
        )

# Standardize zip codes (keeping only the first 5 digits for US zip codes)
zip_columns = ['recipient_zip_code', 'place_of_performance_zip']

for col in zip_columns:
    if col in df_clean.columns:
        # Count before standardization
        valid_before = df_clean[col].notna().sum()
        
        # Standardize: Extract first 5 digits if present
        df_clean[col] = df_clean[col].astype(str).str.extract(r'(\d{5})').iloc[:, 0]
        
        # Count after standardization
        valid_after = df_clean[col].notna().sum()
        
        log_operation(
            'Zip Code Standardization', 
            col, 
            valid_before - valid_after, 
            f"Standardized zip codes in {col}. {valid_before - valid_after} values could not be standardized."
        )

In [ ]:
# Step 5: Range validation for numeric values
# Check and correct out-of-range values in numeric columns

# For financial amounts, ensure they're not negative where that doesn't make sense
for col in numeric_columns:
    if col in df_clean.columns:
        # Count negative values
        negative_values = (df_clean[col] < 0).sum()
        
        if negative_values > 0 and col != 'generated_pragmatic_obligation':  # Some obligations might be negative
            # Flag negative values with a new column
            df_clean[f'{col}_was_negative'] = df_clean[col] < 0
            
            # Replace negative values with 0 or absolute value based on domain knowledge
            df_clean[col] = df_clean[col].abs()
            
            log_operation(
                'Negative Value Correction', 
                col, 
                negative_values, 
                f"Converted {negative_values} negative values to positive in {col}."
            )

In [ ]:
# Step 6: Duplicate detection and handling
# Check for duplicate records based on key fields

# Identify key fields for determining duplicates
key_fields = ['award_id', 'modification_number', 'action_date']

# Find duplicates
if all(field in df_clean.columns for field in key_fields):
    duplicates = df_clean.duplicated(subset=key_fields, keep='first')
    duplicate_count = duplicates.sum()
    
    if duplicate_count > 0:
        # Create a flag for duplicates
        df_clean['is_duplicate'] = duplicates
        
        log_operation(
            'Duplicate Detection', 
            ', '.join(key_fields), 
            duplicate_count, 
            f"Identified {duplicate_count} duplicate records based on key fields."
        )
        
        # Option 1: Keep all records but flag them
        # (Already done above)
        
        # Option 2: Remove duplicates
        # Uncomment the following line to remove duplicates instead of just flagging them
        # df_clean = df_clean[~duplicates].copy()

In [ ]:
# Step 7: Create a data quality summary

# Create a quality summary dataframe from our log
quality_df = pd.DataFrame(quality_log)

# Display the quality summary
print("Data Cleansing Operations Summary:")
display(quality_df)

# Generate basic statistics comparing raw and cleaned data
print("\nData Cleansing Impact Summary:")
print(f"Original shape: {df_raw.shape}")
print(f"Cleaned shape: {df_clean.shape}")

# Compare memory usage
print(f"Original data: {memory_usage(df_raw)}")
print(f"Cleaned data: {memory_usage(df_clean)}")

# Compare null value counts
null_comparison = pd.DataFrame({
    'Original Nulls': df_raw.isnull().sum(),
    'Cleaned Nulls': df_clean.isnull().sum(),
    'Difference': df_raw.isnull().sum() - df_clean.isnull().sum()
})

print("\nNull Value Comparison:")
display(null_comparison)

# Generate some visualizations of the cleansing impact
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
quality_df['rows_affected'].plot(kind='bar')
plt.title('Rows Affected by Each Operation')
plt.xticks(range(len(quality_df)), quality_df.index, rotation=90)

plt.subplot(1, 2, 2)
null_comparison['Difference'].plot(kind='bar')
plt.title('Reduction in Null Values')
plt.tight_layout()
plt.show()

## 4. Save Cleansed Data

Finally, we'll save the cleansed data to a new table in the database or export it to a CSV file for further processing. We'll implement multiple options:

1. Export to a new PostgreSQL table
2. Export to CSV files (potentially in chunks due to size)
3. Export to Parquet format (more efficient for large datasets)

In [ ]:
# Option 1: Save to new database table

# Generate timestamp for table versioning
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
clean_table_name = f"clean_usaspending_awards_{timestamp}"

try:
    # Start timer
    start_time = time.time()
    
    # Write the dataframe to the database
    df_clean.to_sql(
        name=clean_table_name,
        con=engine,
        if_exists='replace',  # Replace if exists
        index=False,
        chunksize=10000,  # Process in chunks to avoid memory issues
        method='multi'  # Use multi-value insert statements for better performance
    )
    
    # Calculate and print execution time
    execution_time = time.time() - start_time
    print(f"Data successfully saved to database table '{clean_table_name}'")
    print(f"Execution time: {execution_time:.2f} seconds")
    
    # Verify the table was created by querying row count
    query = f"SELECT COUNT(*) FROM {clean_table_name}"
    row_count = pd.read_sql(query, engine).iloc[0, 0]
    print(f"Verified table contains {row_count} rows")
    
except Exception as e:
    print(f"Error saving to database: {e}")

In [ ]:
# Option 2: Save to CSV (in chunks due to size)

# Create a directory for the output if it doesn't exist
output_dir = "cleaned_data"
os.makedirs(output_dir, exist_ok=True)

try:
    # Start timer
    start_time = time.time()
    
    # Define chunk size (number of rows per file)
    chunk_size = 1000000  # 1 million rows per file
    
    # Calculate number of chunks needed
    num_chunks = len(df_clean) // chunk_size + (1 if len(df_clean) % chunk_size > 0 else 0)
    
    print(f"Saving data to {num_chunks} CSV files...")
    
    # Save in chunks
    for i in range(num_chunks):
        # Calculate start and end indices
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, len(df_clean))
        
        # Get the chunk
        chunk = df_clean.iloc[start_idx:end_idx]
        
        # Define filename
        filename = f"{output_dir}/clean_usaspending_awards_{timestamp}_part{i+1}.csv"
        
        # Save chunk to CSV
        chunk.to_csv(filename, index=False)
        
        print(f"Saved part {i+1}/{num_chunks}: {filename} ({end_idx - start_idx} rows)")
    
    # Calculate and print execution time
    execution_time = time.time() - start_time
    print(f"All CSV files saved successfully")
    print(f"Execution time: {execution_time:.2f} seconds")
    
except Exception as e:
    print(f"Error saving to CSV: {e}")

In [ ]:
# Option 3: Save to Parquet format (more efficient for large datasets)

try:
    # Start timer
    start_time = time.time()
    
    # Define filename
    parquet_filename = f"{output_dir}/clean_usaspending_awards_{timestamp}.parquet"
    
    # Save to parquet
    df_clean.to_parquet(parquet_filename, index=False, compression='snappy')
    
    # Calculate and print execution time
    execution_time = time.time() - start_time
    print(f"Data successfully saved to parquet file: {parquet_filename}")
    print(f"Execution time: {execution_time:.2f} seconds")
    
    # Get file size
    file_size_bytes = os.path.getsize(parquet_filename)
    file_size_mb = file_size_bytes / (1024 * 1024)
    print(f"File size: {file_size_mb:.2f} MB")
    
except Exception as e:
    print(f"Error saving to parquet: {e}")

## Summary and Next Steps

We have successfully:
1. Connected to the PostgreSQL database
2. Loaded 5,000,000 rows of raw data
3. Performed comprehensive data cleansing operations
4. Saved the cleansed data in multiple formats

### Next Steps:

1. **Analyze the cleansed data**: Now that we have clean data, we can proceed with analysis and visualization.
2. **Automate this pipeline**: Consider scheduling this cleaning process to run regularly.
3. **Optimize for larger datasets**: If needed, explore distributed processing frameworks like Dask or Spark for even larger datasets.
4. **Enhance data validation**: Implement more sophisticated domain-specific validation rules.
5. **Document data quality issues**: Share the data quality logs with the data owners to improve data collection processes.

In [ ]:
# Close all database connections
try:
    cursor.close()
    conn.close()
    print("Database connections closed.")
except:
    pass